# Brandy Roberts Patterns — Seamless Pattern LoRA Trainer

Trains a custom Stable Diffusion XL LoRA on your hand-drawn seamless patterns so the app can generate new repeat-tile patterns for fabric, wallpaper, gift wrap, and digital paper.

**Tuned for surface design:**
- Higher network dim (64) to capture detailed motif shapes
- 12 epochs (more than standard 10) for cleaner motif learning
- Captions emphasize seamless pattern, repeat tile, surface design language

**How to use:**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. When Cell 3 pauses, paste the dataset URL from your app (it starts with `https://` and ends in `.zip` or contains `supabase.co/storage`)
4. Wait 45–70 minutes
5. Copy the final URL and paste it back into Brandy Roberts Patterns app

Trigger word: **brandypattern**

In [ ]:
# Cell 1 — Verify GPU
!nvidia-smi

In [ ]:
# Cell 2 — Install dependencies (~3 min). Ignore the red sentence-transformers warning.
!pip install -q diffusers==0.27.0 transformers==4.40.0 accelerate==0.29.0 peft==0.10.0 bitsandbytes
!pip install -q xformers==0.0.25 datasets safetensors
!git clone https://github.com/kohya-ss/sd-scripts.git
%cd sd-scripts
!pip install -q -r requirements.txt

In [ ]:
# Cell 3 — Get dataset URL
# IMPORTANT: paste the DATASET URL from the app, NOT this Colab page's URL.
import os, requests, zipfile, io

DATASET_URL = input('Paste your training dataset URL from Brandy Roberts Patterns: ').strip()
SESSION_ID = input('Paste your session ID (optional, press Enter to skip): ').strip() or 'session'

if 'colab.research.google.com' in DATASET_URL:
    raise ValueError('That looks like the Colab page URL. You need the DATASET URL from the app instead.')

print(f'Downloading dataset from {DATASET_URL[:80]}...')
r = requests.get(DATASET_URL)
if r.status_code != 200:
    raise ValueError(f'Download failed (status {r.status_code}). The URL may be expired or incorrect.')

# Higher repeats (12) help with pattern learning since shapes need to be very stable
os.makedirs('/content/dataset/12_pattern', exist_ok=True)
try:
    zipfile.ZipFile(io.BytesIO(r.content)).extractall('/content/dataset/12_pattern')
except zipfile.BadZipFile:
    raise ValueError('Downloaded file is not a valid zip. Make sure you copied the dataset URL, not the Colab page URL.')

image_count = len([f for f in os.listdir('/content/dataset/12_pattern') if f.lower().endswith(('.jpg','.jpeg','.png','.webp'))])
print(f'Successfully extracted {image_count} pattern images')

In [ ]:
# Cell 4 — Auto-caption with surface-design language
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

proc = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
mdl = BlipForConditionalGeneration.from_pretrained('Salesforce/blip-image-captioning-base').to('cuda')

for f in os.listdir('/content/dataset/12_pattern'):
    if not f.lower().endswith(('.jpg','.jpeg','.png','.webp')):
        continue
    img = Image.open(f'/content/dataset/12_pattern/{f}').convert('RGB')
    inputs = proc(img, return_tensors='pt').to('cuda')
    cap = proc.decode(mdl.generate(**inputs)[0], skip_special_tokens=True)
    caption_path = f'/content/dataset/12_pattern/{os.path.splitext(f)[0]}.txt'
    # Surface-design caption format: trigger + 'seamless pattern' + BLIP caption + 'flat color hand drawn repeat tile'
    final_cap = f'brandypattern, seamless pattern, {cap}, flat color hand drawn repeat tile, surface design'
    with open(caption_path, 'w') as out:
        out.write(final_cap)
    print(f'{f}: {final_cap}')

print('\nCaptions ready. Trigger word: brandypattern')

In [ ]:
# Cell 5 — Train the SDXL LoRA tuned for patterns (~45-70 min)
# network_dim=64 for richer motif detail, 12 epochs for pattern stability
!accelerate launch --num_cpu_threads_per_process=2 sdxl_train_network.py \
  --pretrained_model_name_or_path='stabilityai/stable-diffusion-xl-base-1.0' \
  --train_data_dir='/content/dataset' \
  --output_dir='/content/output' \
  --output_name='brandypatterns_lora' \
  --resolution='1024,1024' \
  --network_module=networks.lora \
  --network_dim=64 --network_alpha=32 \
  --learning_rate=1e-4 --train_batch_size=1 \
  --max_train_epochs=12 --mixed_precision='fp16' \
  --save_every_n_epochs=3 --save_model_as=safetensors \
  --optimizer_type='AdamW8bit' --xformers \
  --gradient_checkpointing --cache_latents

In [ ]:
# Cell 6 — Upload your trained pattern LoRA
import requests

with open('/content/output/brandypatterns_lora.safetensors', 'rb') as f:
    r = requests.post('https://file.io', files={'file': f})

lora_url = r.json().get('link')
print('\n' + '=' * 60)
print('YOUR BRANDY ROBERTS PATTERNS LORA IS READY')
print('=' * 60)
print(f'\nCopy this URL into the app:\n')
print(lora_url)
print(f'\nTrigger word: brandypattern')
print(f'Example prompt: "brandypattern, seamless pattern of pink strawberries and daisies, flat color hand drawn, pastel pink background"')
print('=' * 60)